In [1]:
import torch.utils.data as data

# DeepGA is an algorithm for evolve a convolutional neural network.


In [3]:
from pathlib import Path
import sys
import os
import torch

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "Radiografias_dxs_pulpares"
CHECKPOINT_DIR = PROJECT_ROOT / "point"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT: ", PROJECT_ROOT)
print("DATA_DIR: ", DATA_DIR)
print("CHECKPOINT_DIR: ", CHECKPOINT_DIR)

PROJECT_ROOT:  c:\Users\MATLAB\Documents\DeepGA_dental\Colab_DeepGA\DeepGA_kvasir
DATA_DIR:  c:\Users\MATLAB\Documents\DeepGA_dental\Colab_DeepGA\DeepGA_kvasir\Radiografias_dxs_pulpares
CHECKPOINT_DIR:  c:\Users\MATLAB\Documents\DeepGA_dental\Colab_DeepGA\DeepGA_kvasir\point


In [4]:
from DeepGA.Operators import *
from DeepGA.EncodingClass import Encoding
from DeepGA.Decoding import *
from DeepGA.DataReader import *
from DeepGA.DistributedTraining import *
from DeepGA.DeepGA import *

In [5]:
%pwd

'c:\\Users\\MATLAB\\Documents\\DeepGA_dental\\Colab_DeepGA\\DeepGA_kvasir'

In [6]:
import torch

print("torch version: ", torch.__version__)
print("cuda available: ", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))
    print("Cuda version: ", torch.version.cuda)
    print("Device count:", torch.cuda.device_count())
else:
    print("No GPU detected by PyThorch")



torch version:  2.5.1
cuda available:  True
GPU:  NVIDIA RTX A1000
Cuda version:  11.8
Device count: 1


In [7]:
import os

data_dir = str(DATA_DIR)

print("Using local dataset at: ", data_dir)
print("Data set exists: ", os.path.exists(data_dir))
print("Classes/folders: ", os.listdir(data_dir))

Using local dataset at:  c:\Users\MATLAB\Documents\DeepGA_dental\Colab_DeepGA\DeepGA_kvasir\Radiografias_dxs_pulpares
Data set exists:  True
Classes/folders:  ['Necrosis Pulpar', 'Previamente iniciado', 'Previamente tratado', 'Pulpa Normal', 'Pulpitis irreversible asintomatica', 'Pulpitis irreversible sintomatica', 'Pulpitis reversible']


# Dataloader

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

def load_radiographs_folder_dataset(
    data_dir,
    image_size=128,
    batch_size=32,
    val_split=0.1,
    test_split=0.1,
    seed=42,
    num_workers=2,
):
    # X-rays are usually grayscale forcing 1 channel for consistency
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])

    full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

    total = len(full_dataset)
    test_size = int(total * test_split)
    val_size = int(total * val_split)
    train_size = total - val_size - test_size

    if train_size <= 0 or val_size <= 0 or test_size <= 0:
        raise ValueError(
            f"Invalid split sizes. total={total}, train={train_size}, val={val_size}, test={test_size}"
        )

    generator = torch.Generator().manual_seed(seed)
    train_set, val_set, test_set = random_split(
        full_dataset, [train_size, val_size, test_size], generator=generator
    )
    pin = torch.cuda.is_available()
    train_dl = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin)
    val_dl = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)
    test_dl = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)

    n_channels = 1
    n_classes = len(full_dataset.classes)
    out_size = image_size

    print("Classes:", full_dataset.classes)
    print("Counts:", total, "train/val/test =", train_size, val_size, test_size)

    return train_dl, val_dl, test_dl, n_channels, n_classes, out_size


In [8]:
%pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


# Data loader Estratificado

In [39]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import StratifiedShuffleSplit
from collections import Counter

def make_stratified_loaders_v2(
    data_dir,
    image_size=128,
    batch_size=32,
    val_split=0.15,
    test_split=0.15,
    seed=42,
    num_workers=2,
):
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])

    ds = datasets.ImageFolder(root=data_dir, transform=transform)
    y = np.array(ds.targets)
    idx = np.arange(len(ds))

    # 1) Split: (train+val) vs test
    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=test_split, random_state=seed)
    trainval_idx, test_idx = next(sss_test.split(idx, y))

    # 2) Split: train vs val (sobre trainval)
    y_trainval = y[trainval_idx]
    val_rel = val_split / (1.0 - test_split)  # val como fracción de lo que quedó
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=val_rel, random_state=seed)
    train_rel, val_rel_idx = next(sss_val.split(trainval_idx, y_trainval))

    train_idx = trainval_idx[train_rel]
    val_idx = trainval_idx[val_rel_idx]

    pin = torch.cuda.is_available()
    train_dl = DataLoader(Subset(ds, train_idx), batch_size=batch_size, shuffle=True,  num_workers=num_workers, pin_memory=pin) #, drop_last=True)
    val_dl   = DataLoader(Subset(ds, val_idx),   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin) #, drop_last=True)
    test_dl  = DataLoader(Subset(ds, test_idx),  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin) #, drop_last=True)

    print("Counts:", len(train_idx), len(val_idx), len(test_idx))
    return train_dl, val_dl, test_dl, 1, len(ds.classes), image_size, ds, train_idx, val_idx, test_idx


In [38]:
from pathlib import Path
from PIL import Image

bad_files=[]

root=Path(data_dir)

for p in root.rglob("*"):
    if p.is_file():
        try:
            with Image.open(p) as img:
                img.verify()
        except Exception as e:
            bad_files.append((str(p), str(e)))

print(f"Archivos malos encontrados: {len(bad_files)}")

for path, err in bad_files[:50]:
    print(path)
    print("->, err")

Archivos malos encontrados: 0


In [18]:
"""
Experimental
Fixme: Revisar salidas de parmetros

"""

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import StratifiedShuffleSplit

def make_stratified_loaders_v2(
    data_dir,
    image_size=128,
    batch_size=32,
    val_split=0.15,
    test_split=0.15,
    seed=42,
    num_workers=2,
):
    # -----------------------------
    # - train_transform: con augmentations
    # - eval_transform : sin augmentations (para val/test)
    # -------------------------
    train_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((image_size, image_size)),

        # AUGMENTATIONS (solo train)
        transforms.RandomRotation(degrees=15),  # ¿rota aleatoriamente? [-15, +15]
        transforms.RandomAffine(
            degrees=0,
            translate=(0.05, 0.05),
            scale=(0.95, 1.05),
            shear=5
        ),
        transforms.RandomHorizontalFlip(p=0.5),  # ¿?

        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])

    eval_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])

    ds_base = datasets.ImageFolder(root=data_dir, transform=None)
    y = np.array(ds_base.targets)
    idx = np.arange(len(ds_base))

    # 1) Split: (train+val) vs test
    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=test_split, random_state=seed)
    trainval_idx, test_idx = next(sss_test.split(idx, y))

    # 2) Split: train vs val (sobre trainval)
    y_trainval = y[trainval_idx]
    val_rel = val_split / (1.0 - test_split)  # val como fracción de lo que quedó
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=val_rel, random_state=seed)
    train_rel, val_rel_idx = next(sss_val.split(trainval_idx, y_trainval))

    train_idx = trainval_idx[train_rel]
    val_idx = trainval_idx[val_rel_idx]

    # -------------------------------------------------------
    # - ds_train: usa train_transform (augmentations)
    # - ds_eval : usa eval_transform (sin augmentations)
    # -------------------------------------------------------
    ds_train = datasets.ImageFolder(root=data_dir, transform=train_transform)
    ds_eval  = datasets.ImageFolder(root=data_dir, transform=eval_transform)

    pin = torch.cuda.is_available()

    train_dl = DataLoader(
        Subset(ds_train, train_idx),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin
    )

    # val/test salen de ds_eval (sin augmentations)
    val_dl = DataLoader(
        Subset(ds_eval, val_idx),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin
    )

    test_dl = DataLoader(
        Subset(ds_eval, test_idx),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin
    )

    print("Counts:", len(train_idx), len(val_idx), len(test_idx))

    #TODO: revisar compatibilidad en el retorno
    return train_dl, val_dl, test_dl, 1, len(ds_base.classes), image_size, ds_base, train_idx, val_idx, test_idx


In [40]:
"""
Muestr la distribucion del estratificado
"""
from collections import Counter
import numpy as np

def show_split_distribution(ds, split_idx, name):
    y = np.array(ds.targets)[split_idx]
    c = Counter(y)
    total = len(split_idx)
    print(f"\n{name} total = {total}")
    for class_id, class_name in enumerate(ds.classes):
        n = c.get(class_id, 0)
        print(f"  {class_name:35s} {n:4d}  ({n/total:.2%})")

train_dl, val_dl, test_dl, n_channels, n_classes, out_size, ds, train_idx, val_idx, test_idx = make_stratified_loaders_v2(
    data_dir=str(DATA_DIR),
    image_size=128,
    batch_size=32,
    val_split=0.15,
    test_split=0.15,
    num_workers=2,
)

show_split_distribution(ds, train_idx, "TRAIN")
show_split_distribution(ds, val_idx,   "VAL")
show_split_distribution(ds, test_idx,  "TEST")


Counts: 1280 275 275

TRAIN total = 1280
  Necrosis Pulpar                      350  (27.34%)
  Previamente iniciado                  87  (6.80%)
  Previamente tratado                  151  (11.80%)
  Pulpa Normal                         216  (16.88%)
  Pulpitis irreversible asintomatica   196  (15.31%)
  Pulpitis irreversible sintomatica    263  (20.55%)
  Pulpitis reversible                   17  (1.33%)

VAL total = 275
  Necrosis Pulpar                       75  (27.27%)
  Previamente iniciado                  19  (6.91%)
  Previamente tratado                   32  (11.64%)
  Pulpa Normal                          47  (17.09%)
  Pulpitis irreversible asintomatica    42  (15.27%)
  Pulpitis irreversible sintomatica     56  (20.36%)
  Pulpitis reversible                    4  (1.45%)

TEST total = 275
  Necrosis Pulpar                       75  (27.27%)
  Previamente iniciado                  19  (6.91%)
  Previamente tratado                   32  (11.64%)
  Pulpa Normal              

### Call DeepGA for evolve CNN´s

In [41]:
'''Defining DeepGA hyperparameters'''
#Convolutional layers
FSIZES = [3, 5, 7, 9] # Odd Sizes Are Preferred
NFILTERS = [8, 16, 32, 64, 128, 256]

#Pooling layers
PSIZES = [2,3] #[2,3,4,5]
PTYPE = ['max', 'avg']

#Fully connected layers
NEURONS = [16, 32, 64, 128, 256] # for big layers

In [42]:
import DeepGA.Operators as ops

ops.FSIZES = FSIZES
ops.NFILTERS = NFILTERS
ops.PSIZES = PSIZES
ops.NEURONS = NEURONS


In [43]:
EXECUTION_ID =710

In [45]:
'''Defining DeepGA hyperparameters'''

#Defining learning rate
lr = 1e-4

#Maximun and minimum numbers of layers to initialize networks
min_conv = 2 # 30
max_conv = 30 # 60
min_full = 1
max_full = 6 # 10
max_params = 9e6
train_epochs = 10 # Epochs to train the best individual found by the GA

'''Genetic Algorithm Parameters'''
cr = 0.7   # Crossover rate
mr = 0.5   # Mutation rate
N = 25      # Population size 20 Se mantuvo en 20
T = 35 #30      # Number of generations
t_size = 5 # tournament size 5
w = 0.1    # penalization weight   0.3

chck_dir = str(CHECKPOINT_DIR)  # Root folder for dumping/loading pickle (checkpoitns)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version (torch): {torch.version.cuda}")
else:
    print("⚠️ Using CPU (no CUDA/GPU detected)")


train_dl, val_dl, test_dl, n_channels, n_classes, out_size, ds, train_idx, val_idx, test_idx = make_stratified_loaders_v2(
    data_dir=str(DATA_DIR),
    image_size=128,
    batch_size=32,
    val_split=0.15,
    test_split=0.15,
    seed=42,
    num_workers=2,
)

# opcional ->>> verificar distribución
show_split_distribution(ds, train_idx, "TRAIN")
show_split_distribution(ds, val_idx,   "VAL")
show_split_distribution(ds, test_idx,  "TEST")

loss_func = nn.CrossEntropyLoss()
execution_ID = EXECUTION_ID #Execution ID for checkpoint, change it for a new execution



✅ Using GPU: NVIDIA RTX A1000
CUDA version (torch): 11.8
Counts: 1280 275 275

TRAIN total = 1280
  Necrosis Pulpar                      350  (27.34%)
  Previamente iniciado                  87  (6.80%)
  Previamente tratado                  151  (11.80%)
  Pulpa Normal                         216  (16.88%)
  Pulpitis irreversible asintomatica   196  (15.31%)
  Pulpitis irreversible sintomatica    263  (20.55%)
  Pulpitis reversible                   17  (1.33%)

VAL total = 275
  Necrosis Pulpar                       75  (27.27%)
  Previamente iniciado                  19  (6.91%)
  Previamente tratado                   32  (11.64%)
  Pulpa Normal                          47  (17.09%)
  Pulpitis irreversible asintomatica    42  (15.27%)
  Pulpitis irreversible sintomatica     56  (20.36%)
  Pulpitis reversible                    4  (1.45%)

TEST total = 275
  Necrosis Pulpar                       75  (27.27%)
  Previamente iniciado                  19  (6.91%)
  Previamente tratado   

This is the train model

In [46]:

results, pop, bestind  = deepGA(execution_ID, True, train_epochs = train_epochs, train_dl=train_dl, val_dl=val_dl,  lr=lr,
                       min_conv=min_conv, max_conv=max_conv, min_full=min_full, max_full=max_full, max_params=max_params,
                       cr=cr, mr=mr, N=N, T=T, t_size=t_size, w=w, device=device, chck_dir=chck_dir,
                       n_channels =  n_channels , n_classes=n_classes, out_size = out_size, loss_func=loss_func)

# training to more epochs

finalEpochs = 150#100
CNNModel = final_evaluation(execution_ID, bestind, train_dl, val_dl, lr, max_params, w, device, finalEpochs, loss_func, chck_dir, n_channels =  n_channels , n_classes=n_classes, out_size = out_size)

Initialize population
0.7961261606060607 0.8836363636363637 8923191
0.8284979747474748 0.8218181818181818 1002455
0.8466249727272727 0.8363636363636363 549207
0.4109025404040404 0.4290909090909091 6775135
0.7359133686868686 0.7454545454545455 3149615
0.6364157929292928 0.6509090909090909 4446215
0.30224662929292934 0.2690909090909091 3594167
0.4579891101010101 0.4072727272727273 770071
0.36087718282828285 0.36363636363636365 5975599
0.4956218292929293 0.4690909090909091 2390399
0.3948056838383838 0.39636363636363636 5572943
0.5140128434343434 0.48727272727272725 2207935
0.8860976757575758 0.8945454545454545 1709391
0.604565796969697 0.6218181818181818 4956351
0.8785537080808081 0.9018181818181819 2977439
0.5648909444444445 0.52 279815
0.6760620797979797 0.7054545454545454 5296231
0.6832116474747475 0.7381818181818182 7303679
0.6767587747474748 0.6618181818181819 1698983
0.45369424141414144 0.4218181818181818 2334791
0.688672083838384 0.6763636363636364 1804967
0.5638857646464647 0.5745

In [26]:
#===================================
#       Minial Test for pipeline
#===================================

train_epochs_test = 1
finalEpochs_test = 2

min_conv_test = 2
max_conv_test = 4
min_full_test = 1
max_full_test = 2
max_params_test = 5e5

cr_test = 0.7
mr_test = 0.5
N_test = 2
T_test = 1
t_size_test = 2
w_test = 0.1

results, pop, bestind= deepGA(
execution_ID,
True,
train_epochs=train_epochs_test,
train_dl=train_dl,
val_dl=val_dl,
lr=lr,
min_conv=min_conv_test,
max_conv=max_conv_test,
min_full=min_full_test,
max_full=max_full_test,
max_params=max_params_test,
cr=cr_test,
mr=mr_test,
N=N_test,
T=T_test,
t_size=t_size_test,
w=w_test,
device=device,
chck_dir=chck_dir,
n_channels=n_classes,
n_classes=n_classes,
out_size=out_size,
loss_func=loss_func
)

CNNModel = final_evaluation(
    execution_ID,
    bestind,
    train_dl,
    val_dl,
    lr,
    max_params_test,
    w_test,
    device,
    finalEpochs_test,
    loss_func,
    chck_dir,
    n_channels=n_channels,
    n_classes=n_classes,
    out_size=out_size
)

Initialize population


RuntimeError: Given groups=1, weight of size [256, 7, 9, 9], expected input[32, 1, 128, 128] to have 7 channels, but got 1 channels instead

In [ ]:
import os
import pandas as pd
import pickle


### save the result in memeoruy in txt file


In [ ]:
"""
save the result in memeoruy in txt file

"""
def save_deepga_run_summary_txt(results, pop, bestind, out_path):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    lines = []
    lines.append("=== DeepGA Run Summary ===\n")

    # ---- results (DataFrame) ----
    lines.append("== results (per generation) ==\n")
    if isinstance(results, pd.DataFrame):
        lines.append(results.to_string(index=True))
    else:
        lines.append(str(results))
    lines.append("\n\n")

    # ---- bestind ----
    lines.append("== bestind (best individual) ==\n")
    try:
        enc = bestind[0]
        lines.append(f"fitness: {bestind[1]}\n")
        lines.append(f"accuracy: {bestind[2]}\n")
        lines.append(f"params: {bestind[3]}\n")
        lines.append("\n-- encoding details --\n")
        # Try to print useful attributes if present
        for attr in ["n_conv", "n_full", "first_level", "second_level"]:
            if hasattr(enc, attr):
                lines.append(f"{attr}: {getattr(enc, attr)}\n")
            else:
                lines.append(f"{attr}: <not present>\n")
    except Exception as e:
        lines.append(f"Could not format bestind: {e}\n")
        lines.append(str(bestind) + "\n")
    lines.append("\n")

    # ---- pop (top-k) ----
    lines.append("== pop (final population) ==\n")
    lines.append(f"population_size: {len(pop)}\n\n")

    # Sort by fitness descending and dump top K
    top_k = min(10, len(pop))
    pop_sorted = sorted(pop, key=lambda x: x[1], reverse=True)

    lines.append(f"-- top {top_k} individuals by fitness --\n")
    for i in range(top_k):
        ind = pop_sorted[i]
        enc = ind[0]
        lines.append(f"\n[{i+1}] fitness={ind[1]}  acc={ind[2]}  params={ind[3]}\n")
        for attr in ["n_conv", "n_full"]:
            if hasattr(enc, attr):
                lines.append(f"  {attr}: {getattr(enc, attr)}\n")

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("".join(lines))

    print("Saved:", out_path)

# Example usage (choose a path)
out_txt = "/content/drive/MyDrive/Workspace/Actividad_DeepGA/DeepGA_kvasir/deepga_summary_708.txt"
save_deepga_run_summary_txt(results, pop, bestind, out_txt)


Saved: /content/drive/MyDrive/Workspace/Actividad_DeepGA/DeepGA_kvasir/deepga_summary_708.txt


### Trying to only train the model

In [ ]:
# Carpeta de checkpoints
chck_dir='/content/drive/MyDrive/Workspace/Actividad_DeepGA/DeepGA_kvasir/point'



In [ ]:

ga_ckpt = os.path.join(chck_dir, f"{EXECUTION_ID}_checkpoint.pkl")

with open(ga_ckpt, "rb") as f:
    state = pickle.load(f)

pop = state["pop"]  # list of [encoding, fitness, accuracy, params]

# Choose criterion:
best_by_fitness = max(pop, key=lambda x: x[1])   # same as DeepGA leader
best_by_accuracy = max(pop, key=lambda x: x[2])  # pure val accuracy

bestind = best_by_fitness  # or best_by_accuracy

print("Loaded bestind:",
      "fitness=", bestind[1],
      "acc=", bestind[2],
      "params=", bestind[3],
      "generation_saved=", state["t"])


Loaded bestind: fitness= 0.45208866000000003 acc= 0.4166666666666667 params= 1145567 generation_saved= 30


In [ ]:
finalEpochs = 150

CNNModel = final_evaluation(
    EXECUTION_ID, bestind,
    train_dl, val_dl,
    lr, max_params, w, device,
    finalEpochs, loss_func, chck_dir,
    n_channels=n_channels, n_classes=n_classes, out_size=out_size,
)


Training final model from best individual
0.47708865999999994 0.4444444444444444 1145567
Execution time:  1006.332914563  seconds
Execution time:  0.27953692071194447  hours
Accuracy:  0.4444444444444444


In [ ]:
state_path = os.path.join(chck_dir, f"Model_Exec_{execution_ID}_Epoch_{finalEpochs}.pt")
torch.save(CNNModel.state_dict(), state_path)
print("✅ Saved state_dict:", state_path)


bestind_path = os.path.join(chck_dir, f"bestind_exec_{execution_ID}.pkl")
with open(bestind_path, "wb") as f:
    pickle.dump(bestind, f)
print("✅ Saved bestind:", bestind_path)

✅ Saved state_dict: point/Model_Exec_707_Epoch_150.pt
✅ Saved bestind: point/bestind_exec_707.pkl


# Cómo cargarlo luego (necesitas reconstruir la arquitectura)

In [ ]:
state_path = os.path.join(chck_dir, f"Model_Exec_{execution_ID}_Epoch_{finalEpochs}.pt")

# Rebuild architecture from bestind
network = decoding(bestind[0], n_channels, out_size, n_classes)
model = CNN(bestind[0], network[0], network[1], network[2]).to(device)

# Load weights
model.load_state_dict(torch.load(state_path, map_location=device))
model.eval()

print("✅ Loaded model with state_dict")

# Carger bestind

In [ ]:
with open(bestind_path, "rb") as f:
    bestind = pickle.load(f)

# TEST

In [ ]:
def evaluate_model(model, data_loader, device, loss_fn=None):
    model.eval()
    total = 0
    correct = 0
    total_loss = 0.0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            if loss_fn is not None:
                loss = loss_fn(outputs, labels)
                total_loss += loss.item() * labels.size(0)

    acc = correct / total if total else 0.0
    avg_loss = (total_loss / total) if (loss_fn is not None and total) else None
    return acc, avg_loss

# ejemplo de uso:
test_acc, test_loss = evaluate_model(CNNModel, test_dl, device, loss_fn=loss_func)
print("Test accuracy:", test_acc)
if test_loss is not None:
    print("Test loss:", test_loss)

Test accuracy: 0.6111111111111112
Test loss: 1.3367948465877109


In [ ]:
print("test examples:", len(test_dl.dataset))


test examples: 36


In [1]:
import torch

PATH_IN = "pointModel_Exec_709_Epoch_150_point.pkl"
PATH_OUT = "pointModel_Exec_709_Epoch_150_point_cpu.pkl"

obj = torch.load(PATH_IN, weights_only=False)

if hasattr(obj, "to"):
    obj = obj.to("cpu")

if isinstance(obj, dict):
    for k, v in obj.items():
        if hasattr(v, "to"):
            obj[k] = v.to("cpu")

torch.save(obj, PATH_OUT)
print(f"Guardado en CPU: {PATH_OUT}")


RuntimeError: Invalid magic number; corrupt file?